# Figure S9 A & B — CMAP pattern × cell-type / layer correlation dot plots

Same method as the CoGAPS analysis notebook
(`Code/Figure_2/CoGAPS_CMAP/n30_cogaps_analyzePats_updated.ipynb`, "Correlate with metadata"):
Spearman correlation of each CoGAPS pattern's per-spot weight with one-hot cell-type / layer labels, i.e.
`patterns.merge(pd.get_dummies(annotation)).corr`, restricted to the 3 CMAP patterns.

* **A** — CMAP_PCL / CMAP_IGL / CMAP_IGL_IMM × cell type (`Cell_Type`)
* **B** — CMAP_IGL / CMAP_IGL_IMM × granular layer (EGL = external, IGL = internal)

Dot **colour = correlation** (Corr., −0.3…0.3, RdBu), dot **size = |r|** (0.1/0.3/0.5).
Panels **C–E** (spatial maps) are server-side image assets — not generated here.

Inputs (repo-local, relative): `Supplementary_Datasets/Data_S4_CoGAPS_Pattern_Weights.csv`, `Supplementary_Datasets/Data_S4_Spot_Metadata.csv`.
Pattern IDs: Pattern_27=CMAP_PCL, Pattern_17=CMAP_IGL, Pattern_16=CMAP_IGL_IMM.

In [1]:
import warnings; warnings.filterwarnings("ignore")  # clean render: hide non-fatal warnings
import os
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, matplotlib as mpl

def find_s9(fname, subdir=".", max_up=6):
    d="."
    for _ in range(max_up+1):
        c=os.path.join(d,"Supplementary_Datasets",subdir,fname)
        if os.path.exists(c): return c
        d=os.path.join(d,"..")
    raise FileNotFoundError(fname)

patterns = pd.read_csv(find_s9("Data_S4_CoGAPS_Pattern_Weights.csv"), index_col=0).T   # spots x patterns
meta     = pd.read_csv(find_s9("Data_S4_Spot_Metadata.csv"), index_col=0, low_memory=False)
patterns = patterns.loc[meta.index]
PMAP = {"Pattern_27":"CMAP_PCL", "Pattern_17":"CMAP_IGL", "Pattern_16":"CMAP_IGL_IMM"}

def corr_block(col, repl, order, pats):
    lab = meta[col].replace(repl)
    dummies = pd.get_dummies(lab)
    C = patterns[pats].join(dummies).corr(method="spearman").loc[pats, dummies.columns]
    C.index = [PMAP[p] for p in pats]
    return C[[c for c in order if c in C.columns]]

A = corr_block("Cell_Type", {"Granular":"Granule","Bergmann":"Bergmann Glia"},
               ["Granule","PLI-3","Purkinje","Bergmann Glia","PLI-1/2","WM","MLI-1","MLI-2"],
               ["Pattern_27","Pattern_17","Pattern_16"])
B = corr_block("granular_layer", {"external":"EGL","internal":"IGL"}, ["EGL","IGL"],
               ["Pattern_17","Pattern_16"])
A.round(3)

,Granule,PLI-3,Purkinje,Bergmann Glia,PLI-1/2,WM,MLI-1,MLI-2
CMAP_PCL,-0.308,-0.128,0.351,0.090,-0.041,0.000,0.154,0.087
CMAP_IGL,0.344,0.037,-0.153,-0.020,0.115,-0.320,-0.090,-0.076
CMAP_IGL_IMM,0.080,0.140,-0.156,0.038,0.203,-0.215,-0.071,-0.024


In [2]:
NORM = mpl.colors.Normalize(-0.6, 0.6); CMAP = plt.cm.RdBu_r; SZ = 1200
def _wrap(s):   return "Bergmann\nGlia" if s == "Bergmann Glia" else s
def _rowlab(s): return s.replace("CMAP_", "CMAP\n_")

def dotplot(M, stem, figsize):
    rows, cols = list(M.index), list(M.columns); nr, nc = len(rows), len(cols)
    fig, ax = plt.subplots(figsize=figsize)
    for yi, r in enumerate(rows):
        ax.plot([-.5, nc-.5], [yi, yi], color="0.88", lw=0.7, zorder=0)
        for xi, c in enumerate(cols):
            v = M.loc[r, c]
            ax.scatter(xi, yi, s=abs(v)*SZ+8, c=[CMAP(NORM(v))],
                       edgecolors="0.35", linewidths=0.3, zorder=3)
    ax.set_yticks(range(nr)); ax.set_yticklabels([_rowlab(r) for r in rows], fontsize=9)
    ax.set_xticks(range(nc)); ax.set_xticklabels([_wrap(c) for c in cols], fontsize=8)
    ax.set_xlim(-0.7, nc-0.3); ax.set_ylim(-0.7, nr-0.3); ax.invert_yaxis()
    for sp in ax.spines.values(): sp.set_color("0.4")
    ax.tick_params(length=0)
    sm = mpl.cm.ScalarMappable(norm=NORM, cmap=CMAP); sm.set_array([])
    cb = fig.colorbar(sm, ax=ax, fraction=0.05, pad=0.02, ticks=[-0.3, 0, 0.3], aspect=10)
    cb.ax.set_title("Spearman\ncorrelation", fontsize=7.5, pad=4); cb.ax.tick_params(labelsize=7)
    h = [ax.scatter([], [], s=rv*SZ+8, c="0.6", edgecolors="0.35", linewidths=0.3, label=str(rv))
         for rv in (0.1, 0.3, 0.5)]
    ax.legend(handles=h, title="rho", loc="center left", bbox_to_anchor=(1.14, 0.5),
              frameon=False, labelspacing=1.3, fontsize=7, title_fontsize=8, borderpad=0)
    fig.savefig(stem+".pdf", bbox_inches="tight")
    plt.show(); plt.close(fig)

dotplot(A, "Figure_S9A_Pattern_CellType", (6.4, 2.6))
dotplot(B, "Figure_S9B_Pattern_Layer",   (3.5, 2.2))

/var/folders/x6/5n5h1qyn3x50lrl84lh5p2vc0000gn/T/ipykernel_20304/1676638029.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
 plt.show; plt.close(fig)
/var/folders/x6/5n5h1qyn3x50lrl84lh5p2vc0000gn/T/ipykernel_20304/1676638029.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
 plt.show; plt.close(fig)
